In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [2]:
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda


In [4]:
clip_df = pd.read_csv('../clips_metadata.csv',sep=",")

N_MELS = 128
SR = 22050
DURATION = 5
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 41

print(f'Total clips: {len(clip_df)}')              # ← clip_df not df
print(f'Classes: {clip_df["species"].nunique()}')   # ← species not emotion

Total clips: 49682
Classes: 41


In [6]:
le = LabelEncoder()
clip_df['label'] = le.fit_transform(clip_df['species'])

In [8]:
class AviaNetDataset(Dataset):
    def __init__(self, clip_df, augment_data=False):
        self.clip_df = clip_df.reset_index(drop=True)
        self.augment_data = augment_data
    def __len__(self):
        return len(self.clip_df)
    def __getitem__(self, idx):
        row = self.clip_df.iloc[idx]
        file_path = row['path']

        y, sr = librosa.load(file_path, sr=SR, duration=DURATION)

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        if self.augment_data:
            y = self.augment(y, sr)

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)
        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(row['label'], dtype=torch.long)

        return tensor, label

    def augment(self, y, sr):
        noise = np.random.randn(len(y)) * 0.005
        y = y + noise
        shift = np.random.randint(0, max(1, sr // 4))
        y = np.roll(y, shift)
        steps = np.random.uniform(-2, 2)
        y = librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

        return y

print('Dataset class defined')


Dataset class defined


In [9]:
train_df, test_df = train_test_split(
    clip_df, test_size=0.2, random_state=42, stratify=clip_df['label']
)

train_dataset = AviaNetDataset(train_df, augment_data=True)
test_dataset = AviaNetDataset(test_df, augment_data=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

batch_x, batch_y = next(iter(train_loader))
print(f'Batch X shape: {batch_x.shape}')
print(f'Batch y shape: {batch_y.shape}')

Train samples: 39745
Test samples:  9937
Train batches: 1243
Test batches:  311
Batch X shape: torch.Size([32, 1, 128, 216])
Batch y shape: torch.Size([32])
